# 05 - Figures académiques pour le rapport

Génère les figures publication-ready à partir du CSV de test final v2.

Sortie: `outputs/msd_implementation/resnet50_wide_crop/figures/*.pdf` (vectoriel pour LaTeX) et `*.png` (preview).

Chaque figure est conçue pour tenir en colonne IJCAI (3.375" de large) ou pleine largeur (7" de large). Toutes utilisent la même charte graphique sobre.

In [ ]:
#@title Setup (imports, style, paths)
from __future__ import annotations
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
from colab.drive_paths import output_dir

# IJCAI single-column width = 3.375 in, two-column width = 7.0 in
COL_W = 3.375
FULL_W = 7.0

# Sober academic style: black axes, muted palette, sans-serif
PALETTE = {
    "TP": "#2E7D32",   # green
    "FP": "#C62828",   # red
    "TN": "#1565C0",   # blue
    "FN": "#EF6C00",   # orange
    "primary": "#37474F",
    "secondary": "#90A4AE",
    "accent": "#E65100",
}

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "lines.linewidth": 1.5,
    "pdf.fonttype": 42,  # editable text in Illustrator
    "ps.fonttype": 42,
})

# Walk up from CWD looking for a marker file at the repo root. Robust to
# launching from repo root, from the notebook folder via nbconvert, or from
# /content/INF8225_Projet on Colab.
def _find_repo_root() -> Path:
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "colab" / "setup.py").exists():
            return p
    return Path.cwd()

REPO_ROOT = _find_repo_root()

# Locate CSV: try local Mac path first, then repo path, then Colab path
CANDIDATE_CSV_PATHS = [
    Path("/Users/moradbelmelih/Downloads/dice_final_report_msd_recall_3slice_v2-2.csv"),
    REPO_ROOT / "outputs/msd_implementation/resnet50_wide_crop/metrics/dice_final_report_resnet50_wide_crop.csv",
    Path("/content/INF8225_Projet/outputs/msd_implementation/resnet50_wide_crop/metrics/dice_final_report_resnet50_wide_crop.csv"),
]
CSV_PATH = next((p for p in CANDIDATE_CSV_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(f"CSV introuvable parmi : {CANDIDATE_CSV_PATHS}")
print("CSV utilisé :", CSV_PATH)

# Output directory: persistent Drive/repo output tree.
FIG_DIR = output_dir("msd_implementation", "resnet50_wide_crop", "figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root      :", REPO_ROOT)
print("Sortie figures :", FIG_DIR.resolve())

def save_fig(fig, name: str):
    """Save figure as PDF (vector) and PNG (preview) under FIG_DIR."""
    fig.savefig(FIG_DIR / f"{name}.pdf")
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"  → {FIG_DIR / name}.{{pdf,png}}")

In [ ]:
#@title Load CSV and compute base stats
df = pd.read_csv(CSV_PATH)
df['score'] = df['best_candidate_score']
df['predicted'] = df['n_selected'] > 0

# Parse patient and slice from filename
_re = re.compile(r'pancreas_(\d+)_s(\d+)\.png$')
df['patient'] = df['file_name'].str.extract(_re)[0].astype(int)
df['slice']   = df['file_name'].str.extract(_re)[1].astype(int)

# Categorize
def category(row):
    if row['has_tumor'] and row['predicted']:
        return 'TP'
    if row['has_tumor'] and not row['predicted']:
        return 'FN'
    if not row['has_tumor'] and row['predicted']:
        return 'FP'
    return 'TN'
df['category'] = df.apply(category, axis=1)

# Confusion matrix and metrics
tp = (df.category == 'TP').sum()
fp = (df.category == 'FP').sum()
tn = (df.category == 'TN').sum()
fn = (df.category == 'FN').sum()
sens = tp/(tp+fn) if (tp+fn) else 0
spec = tn/(tn+fp) if (tn+fp) else 0
prec = tp/(tp+fp) if (tp+fp) else 0
f1   = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0
dice_pos = df.loc[df.has_tumor, 'final_dice'].mean()

print(f"N images        : {len(df)}")
print(f"  positives     : {df.has_tumor.sum()}")
print(f"  négatives     : {(~df.has_tumor).sum()}")
print()
print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")
print(f"Sensibilité    : {sens:.3f}")
print(f"Spécificité    : {spec:.3f}")
print(f"Précision      : {prec:.3f}")
print(f"F1             : {f1:.3f}")
print(f"DICE moy (TP)  : {dice_pos:.3f}")

In [ ]:
#@title Figure 1 — Distribution des scores ResNet par catégorie
# Idée : montrer la séparabilité TP/FP et l'overlap FN/TN qui motive l'analyse.

fig, ax = plt.subplots(figsize=(COL_W, 2.4))

bins = np.linspace(0, 1, 26)
for cat in ['TN', 'FN', 'FP', 'TP']:
    sub = df[df.category == cat]
    ax.hist(sub['score'], bins=bins, alpha=0.65, label=f'{cat} (n={len(sub)})',
            color=PALETTE[cat], edgecolor='white', linewidth=0.4)

# Vertical line at the operating threshold (implicit from n_selected)
tau_inferred = df.loc[df.predicted, 'score'].min()
ax.axvline(tau_inferred, color=PALETTE['primary'], linestyle='--', linewidth=1.0,
           label=f'τ = {tau_inferred:.2f}')

ax.set_xlabel('Score ResNet ensemble')
ax.set_ylabel('Nombre de coupes')
ax.set_xlim(0, 1)
ax.legend(loc='upper center', frameon=False, ncol=1, fontsize=7)
ax.set_title('Distribution des scores par catégorie de prédiction', pad=6)
fig.tight_layout()
save_fig(fig, 'fig1_score_distribution')
plt.show()

In [ ]:
#@title Figure 2 — ROC et Précision-Rappel
# Idée : courbes standard ML, AUC reporté en légende.

from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

y = df.has_tumor.values.astype(int)
s = df.score.values

fpr, tpr, _ = roc_curve(y, s)
roc_auc = auc(fpr, tpr)
prec_curve, rec_curve, _ = precision_recall_curve(y, s)
ap = average_precision_score(y, s)

fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.8))

ax = axes[0]
ax.plot(fpr, tpr, color=PALETTE['accent'], linewidth=1.6,
        label=f'Pipeline (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color=PALETTE['secondary'], linestyle=':', linewidth=1.0,
        label='Hasard')
# Mark the operating point
ax.scatter([1-spec], [sens], s=40, color=PALETTE['primary'], zorder=5,
           label=f'Point opératoire (τ={tau_inferred:.2f})')
ax.set_xlabel('1 − Spécificité (FPR)')
ax.set_ylabel('Sensibilité (TPR)')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(loc='lower right', frameon=False)
ax.set_title('Courbe ROC', pad=6)
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.plot(rec_curve, prec_curve, color=PALETTE['accent'], linewidth=1.6,
        label=f'Pipeline (AP = {ap:.3f})')
# Baseline = positive rate
pos_rate = y.mean()
ax.axhline(pos_rate, color=PALETTE['secondary'], linestyle=':', linewidth=1.0,
           label=f'Hasard (pos rate = {pos_rate:.2f})')
ax.scatter([sens], [prec], s=40, color=PALETTE['primary'], zorder=5,
           label=f'Point opératoire')
ax.set_xlabel('Sensibilité (Recall)')
ax.set_ylabel('Précision')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(loc='lower left', frameon=False)
ax.set_title('Courbe Précision–Rappel', pad=6)
ax.grid(True, alpha=0.25)

fig.tight_layout()
save_fig(fig, 'fig2_roc_pr')
plt.show()

In [ ]:
#@title Figure 3 — Sweep du seuil τ
# Idée : montrer le compromis sensibilité/spécificité/F1 en fonction de τ.

taus = np.linspace(0.0, 1.0, 101)
rows = []
for t in taus:
    pred = df.score >= t
    has = df.has_tumor
    tp_t = ((pred) & (has)).sum()
    fp_t = ((pred) & (~has)).sum()
    tn_t = ((~pred) & (~has)).sum()
    fn_t = ((~pred) & (has)).sum()
    rows.append({
        'tau': t,
        'sens': tp_t/(tp_t+fn_t) if (tp_t+fn_t) else 0,
        'spec': tn_t/(tn_t+fp_t) if (tn_t+fp_t) else 0,
        'prec': tp_t/(tp_t+fp_t) if (tp_t+fp_t) else 0,
        'f1':   2*tp_t/(2*tp_t+fp_t+fn_t) if (2*tp_t+fp_t+fn_t) else 0,
    })
sweep = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(COL_W, 2.4))
ax.plot(sweep.tau, sweep.sens, label='Sensibilité', color=PALETTE['TP'], linewidth=1.4)
ax.plot(sweep.tau, sweep.spec, label='Spécificité', color=PALETTE['TN'], linewidth=1.4)
ax.plot(sweep.tau, sweep.prec, label='Précision',  color=PALETTE['FP'], linewidth=1.4, linestyle='--')
ax.plot(sweep.tau, sweep.f1,   label='F1',          color=PALETTE['primary'], linewidth=1.8)
ax.axvline(tau_inferred, color=PALETTE['secondary'], linestyle=':', linewidth=1.0,
           label=f'τ retenu = {tau_inferred:.2f}')

best_tau_idx = sweep.f1.idxmax()
ax.scatter([sweep.tau.iloc[best_tau_idx]], [sweep.f1.iloc[best_tau_idx]],
           color=PALETTE['accent'], s=30, zorder=5,
           label=f'F1 max = {sweep.f1.iloc[best_tau_idx]:.2f}')

ax.set_xlabel('Seuil τ sur le score ResNet')
ax.set_ylabel('Métrique')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(loc='lower left', frameon=False, ncol=2, fontsize=7)
ax.set_title('Compromis classification vs τ', pad=6)
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_fig(fig, 'fig3_threshold_sweep')
plt.show()

In [ ]:
#@title Figure 4 — Matrice de confusion + DICE par catégorie
# Idée : compact, deux panneaux : confusion + qualité de segmentation.

fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.8))

# Panel A: confusion matrix
ax = axes[0]
cm = np.array([[tp, fn], [fp, tn]])
im = ax.imshow(cm, cmap='Blues', aspect='equal', vmin=0, vmax=cm.max()*1.1)
for i in range(2):
    for j in range(2):
        col = 'white' if cm[i,j] > cm.max()*0.55 else PALETTE['primary']
        ax.text(j, i, f'{cm[i,j]}', ha='center', va='center',
                fontsize=14, fontweight='bold', color=col)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Tumeur prédite', 'Pas de tumeur'])
ax.set_yticklabels(['Tumeur réelle', 'Pas de tumeur'])
ax.set_title(f'Matrice de confusion (F1 = {f1:.3f})', pad=6)
ax.set_xlabel('')
ax.set_ylabel('')
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)

# Panel B: DICE distribution per category (TP/FN/all positives)
ax = axes[1]
data_box = [
    df.loc[df.category == 'TP', 'final_dice'].values,
    df.loc[df.has_tumor, 'final_dice'].values,
]
bp = ax.boxplot(data_box, vert=True, patch_artist=True, widths=0.55,
                showmeans=True,
                meanprops={'marker':'D','markerfacecolor':'white','markeredgecolor':PALETTE['primary'],'markersize':5},
                medianprops={'color':PALETTE['primary'],'linewidth':1.4},
                boxprops={'facecolor':PALETTE['secondary'],'edgecolor':PALETTE['primary'],'alpha':0.4},
                whiskerprops={'color':PALETTE['primary']},
                capprops={'color':PALETTE['primary']},
                flierprops={'marker':'o','markersize':3,'markerfacecolor':PALETTE['secondary'],'markeredgecolor':'none','alpha':0.6})
# Color each box differently
bp['boxes'][0].set_facecolor(PALETTE['TP']); bp['boxes'][0].set_alpha(0.4)
bp['boxes'][1].set_facecolor(PALETTE['secondary']); bp['boxes'][1].set_alpha(0.4)

ax.set_xticks([1, 2])
ax.set_xticklabels([f'TP\n(n={tp})', f'Toutes positives\n(n={tp+fn})'])
ax.set_ylabel('DICE')
ax.set_ylim(-0.02, 1.02)
ax.set_title('Qualité de segmentation MedSAM', pad=6)
ax.grid(True, axis='y', alpha=0.25)

fig.tight_layout()
save_fig(fig, 'fig4_confusion_dice')
plt.show()

In [ ]:
#@title Figure 5 — Profil par patient (heatmap des scores)
# Idée : montrer visuellement les patients faciles vs difficiles, et l'effet du seuil.

# Build patient × slice matrix (limited to patients with tumor)
tumor_patients = sorted(df.loc[df.has_tumor, 'patient'].unique())
all_slices = sorted(df.slice.unique())

# We want a focused view: patients that have tumor slices in the test
focused = df[df.patient.isin(tumor_patients)].copy()
focused = focused.sort_values(['patient','slice'])

fig, ax = plt.subplots(figsize=(FULL_W, 4.2))

y_positions = {p: i for i, p in enumerate(tumor_patients)}

# Plot one row per patient: scatter at slice index, color by score, marker by has_tumor
for _, r in focused.iterrows():
    y = y_positions[r['patient']]
    color = plt.cm.RdYlGn(min(r['score'], 1.0))
    marker = 's' if r['has_tumor'] else 'o'
    edge = PALETTE['primary'] if r['predicted'] else 'none'
    lw = 1.2 if r['predicted'] else 0
    ax.scatter(r['slice'], y, s=70, c=[color], marker=marker,
               edgecolors=edge, linewidths=lw)

ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels([f'P{p:03d}' for p in tumor_patients], fontsize=7)
ax.set_xlabel('Indice de coupe (slice)')
ax.set_ylabel('Patient')
ax.set_title('Score ResNet par patient et coupe — patients avec tumeur uniquement', pad=6)
ax.grid(True, axis='x', alpha=0.25)

# Colorbar
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=mpl.colors.Normalize(vmin=0, vmax=1))
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Score ResNet', fontsize=8)
cbar.ax.tick_params(labelsize=7)

# Custom legend for marker shape and edge
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor=PALETTE['secondary'], markeredgecolor='none',
           markersize=8, label='Coupe avec tumeur'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PALETTE['secondary'], markeredgecolor='none',
           markersize=8, label='Coupe saine'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=PALETTE['secondary'],
           markeredgecolor=PALETTE['primary'], markersize=8, label='Détectée par le pipeline'),
]
ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=3, frameon=False, fontsize=8)
fig.tight_layout()
save_fig(fig, 'fig5_patient_profile')
plt.show()

In [ ]:
#@title Figure 6 — Comparaison à travers les itérations
# Idée : barplot horizontal comparant baseline 2D, 3-slice v1 et 3-slice v2.
# Les chiffres 2D et v1 sont issus des CSV antérieurs ou hardcodés depuis le rapport.

iter_data = pd.DataFrame([
    {'Pipeline': 'DINO + MedSAM\n(baseline 2D)',     'TP': 31, 'FP': 10, 'TN': 40, 'FN': 19,  'F1': 0.683},
    {'Pipeline': '+ ResNet-18 ensemble\n(2D)',        'TP': 31, 'FP': 10, 'TN': 40, 'FN': 19,  'F1': 0.683},
    {'Pipeline': '+ contexte 3-slice\n(v1)',          'TP': 27, 'FP':  2, 'TN': 48, 'FN': 23,  'F1': 0.683},
    {'Pipeline': '+ ResNet-50 + crop=30\n(v2)',       'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,  'F1': f1},
])
iter_data['Sens'] = iter_data.TP / (iter_data.TP + iter_data.FN)
iter_data['Spec'] = iter_data.TN / (iter_data.TN + iter_data.FP)
iter_data['Prec'] = iter_data.TP / (iter_data.TP + iter_data.FP)

fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 3.0), sharey=True)

y_pos = np.arange(len(iter_data))

# Panel A: F1 / Sens / Spec / Prec
ax = axes[0]
h = 0.18
for i, (col, color) in enumerate([('Sens', PALETTE['TP']),
                                  ('Spec', PALETTE['TN']),
                                  ('Prec', PALETTE['FP']),
                                  ('F1',   PALETTE['primary'])]):
    ax.barh(y_pos + (i-1.5)*h, iter_data[col], height=h, color=color, label=col, alpha=0.85)
    for j, v in enumerate(iter_data[col]):
        ax.text(v + 0.01, y_pos[j] + (i-1.5)*h, f'{v:.2f}', va='center', fontsize=6)
ax.set_yticks(y_pos)
ax.set_yticklabels(iter_data['Pipeline'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Métrique')
ax.set_xlim(0, 1.05)
ax.legend(loc='lower right', frameon=False, fontsize=7, ncol=2)
ax.set_title('Métriques de classification', pad=6)
ax.grid(True, axis='x', alpha=0.25)

# Panel B: TP/FP/TN/FN counts (stacked bar)
ax = axes[1]
bottoms = np.zeros(len(iter_data))
for col, color in [('TP', PALETTE['TP']), ('FN', PALETTE['FN']),
                   ('FP', PALETTE['FP']), ('TN', PALETTE['TN'])]:
    ax.barh(y_pos, iter_data[col], left=bottoms, color=color, label=col, alpha=0.85, height=0.55)
    for j, v in enumerate(iter_data[col]):
        if v > 2:
            ax.text(bottoms[j] + v/2, y_pos[j], str(v), ha='center', va='center', fontsize=7,
                    color='white', fontweight='bold')
    bottoms += iter_data[col].values
ax.set_xlim(0, 100)
ax.invert_yaxis()
ax.set_xlabel('Nombre de coupes (sur 100)')
ax.legend(loc='lower right', frameon=False, fontsize=7, ncol=4)
ax.set_title('Décomposition des décisions', pad=6)
ax.grid(True, axis='x', alpha=0.25)

fig.tight_layout()
save_fig(fig, 'fig6_iterations_comparison')
plt.show()

In [ ]:
#@title Récapitulatif : table chiffrée à recopier dans le rapport
summary = pd.DataFrame([
    {'Métrique': 'TP', 'Valeur': tp},
    {'Métrique': 'FP', 'Valeur': fp},
    {'Métrique': 'TN', 'Valeur': tn},
    {'Métrique': 'FN', 'Valeur': fn},
    {'Métrique': 'Sensibilité', 'Valeur': round(sens, 3)},
    {'Métrique': 'Spécificité', 'Valeur': round(spec, 3)},
    {'Métrique': 'Précision',   'Valeur': round(prec, 3)},
    {'Métrique': 'F1',          'Valeur': round(f1, 3)},
    {'Métrique': 'DICE moy. positives', 'Valeur': round(dice_pos, 3)},
    {'Métrique': 'DICE médian positives', 'Valeur': round(df.loc[df.has_tumor,'final_dice'].median(), 3)},
    {'Métrique': 'AUC ROC', 'Valeur': round(roc_auc, 3)},
    {'Métrique': 'AP',      'Valeur': round(ap, 3)},
])
print(summary.to_string(index=False))

# Save the summary alongside figures
summary.to_csv(FIG_DIR / 'fig_summary_metrics.csv', index=False)
print(f'\n→ {FIG_DIR / "fig_summary_metrics.csv"}')

print('\nFigures sauvegardées :')
for p in sorted(FIG_DIR.glob('fig*.pdf')):
    print(' ', p.name)

In [ ]:
#@title Vérifier distributions de scores TP / TN / FN

has = df["has_tumor"].astype(bool)
dice = df["final_dice"].astype(float)

# - scan tumoral détecté si Dice > 0
# - scan non tumoral correctement vide si Dice == 1
tp = int((has & (dice > 0)).sum())
fn = int((has & (dice == 0)).sum())
tn = int((~has & (dice == 1)).sum())
fp = int((~has & (dice < 1)).sum())

tp_mask = has & (dice > 0)
fn_mask = has & (dice == 0)
tn_mask = (~has) & (dice == 1)
fp_mask = (~has) & (dice < 1)

for col in ["best_candidate_score", "best_resnet_prob", "best_dino_score"]:
    if col not in df.columns:
        continue
    print("\n", col)
    print("TP median:", df.loc[tp_mask, col].median())
    print("TN median:", df.loc[tn_mask, col].median())
    print("FN median:", df.loc[fn_mask, col].median())
    print("FP median:", df.loc[fp_mask, col].median())
